# Exploratory Data Analysis

Before building any models, we need to understand the structure and characteristics of our cleaned dataset. This notebook investigates:

1. **Speech length distributions** — Do war-era and peacetime speeches differ in length? Should we normalize or truncate?
2. **Vocabulary patterns** — What are the most frequent words across classes? Do the same words dominate both, or are there distinguishing terms?
3. **Class balance** — How severe is the imbalance, and what does it look like after cleaning?

These findings will directly inform our modeling decisions in the next notebook.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import tabulate
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# Add project root to path for local imports
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from words_of_war.visualization import (
    plot_speech_length_distribution,
    plot_class_distributions,
    plot_class_boxplots,
    plot_overall_top_words,
    plot_top_words,
)

In [ ]:
df = pd.read_csv('../data/Speeches_War_Clean.csv')

# Compute text length if not already present
if 'Text Length' not in df.columns:
    df['Text Length'] = df['Transcript'].apply(lambda x: len(str(x).split()))

## Initial Inspection

A quick look at the cleaned dataset to confirm the schema and verify that our target variable (`War`) and computed `Text Length` column are present.

In [ ]:
df.head()

## Speech Length Distributions

Speech length is one of the simplest features we can examine. If war-era speeches are systematically longer or shorter than peacetime speeches, length alone could be a weak signal — or a confound we need to account for.

In [ ]:
fig = plot_speech_length_distribution(df)
plt.show()

The overall distribution is **heavily right-skewed** — most speeches fall under ~15,000 words, but a long tail of outliers extends well beyond 20,000. This suggests that a few exceptionally long addresses (likely State of the Union or annual messages) dominate the upper range.

Next, let's split this by class to see if the skew differs between war and peacetime speeches.

In [ ]:
fig = plot_class_distributions(df)
plt.show()

Both classes exhibit similar right-skewed shapes, but the **peacetime (War=0) distribution has a heavier tail** — it contains the longest speeches in the corpus. War-era speeches are more concentrated in the short-to-mid range. This is worth noting but likely reflects the larger sample size in the peacetime class rather than a meaningful rhetorical pattern.

The boxplots below give a cleaner comparison of the central tendencies and spread.

In [ ]:
fig = plot_class_boxplots(df)
plt.show()

The percentile breakdown confirms what the boxplots show:

In [ ]:
grouped_stats = df.groupby('War')['Text Length'].describe(percentiles=[0.25, 0.5, 0.75])

# Extracting specific percentiles and renaming columns
grouped_stats = grouped_stats[['min', '25%', '50%', '75%', 'max']].rename(columns={'25%': '25th_percentile', '50%': 'median', '75%': '75th_percentile'})

print(grouped_stats)

**Key takeaway:** The medians are nearly identical, suggesting speech length is **not a strong discriminator** between classes. The main difference is in the extremes — peacetime speeches have a much higher max. Since our BERT vectorizer truncates to 256 tokens, these outliers won't affect modeling directly, but it validates our choice of a transformer-based approach that focuses on semantic content rather than length-based features.

## Vocabulary Analysis

Next, we examine word frequency to understand what the models will actually "see" in the text. If the same high-frequency words dominate both classes, the models will need to look beyond surface-level word counts to find discriminating patterns.

In [ ]:
fig = plot_overall_top_words(df)
plt.show()

The top 25 words are overwhelmingly **stopwords and function words** ("the", "of", "and", "to", "in"). This is expected for political speech — the rhetorical signal that distinguishes war-era rhetoric lies in less frequent but more semantically loaded terms.

This observation motivates our choice of **BERT embeddings** over a simple bag-of-words approach. BERT captures contextual meaning and word relationships, allowing the model to learn from how words are *used together* rather than how often individual words appear.

Now let's compare the top words side by side, with **words common to both classes highlighted in green**. If the top vocabulary is nearly identical across classes, it reinforces our need for contextual embeddings rather than frequency-based features.

In [ ]:
fig = plot_top_words(df)
plt.show()

As expected, the **vast majority of top words are shared** between classes (shown in green). The high-frequency vocabulary of presidential speeches is largely class-agnostic — both war and peacetime presidents rely on the same functional language ("the", "of", "states", "government", "united").

This confirms that a simple word-count or TF-IDF approach would struggle to differentiate between classes. The discriminating signal must come from **contextual patterns** — how words are combined, the rhetorical framing, and the semantic nuance of specific passages. This is precisely where BERT-based embeddings and attention mechanisms excel.

## Conclusions

This exploratory analysis reveals several key insights that shape our modeling strategy:

1. **Speech length is not a discriminator.** War-era and peacetime speeches have nearly identical median lengths (~15,200 words). Models should not rely on length as a feature.

2. **High-frequency vocabulary is shared across classes.** The most common words are stopwords and generic political language. Discriminating between classes requires capturing semantic context, not word frequency.

3. **BERT embeddings are well-motivated.** Given the above findings, a contextual embedding approach (BERT) that captures meaning from word relationships is better suited than bag-of-words or TF-IDF representations.

4. **Class imbalance remains a concern.** The raw dataset has a ~92/8 split (peacetime/war). The SMOTE-based resampling applied in the cleaning notebook addresses this, but model evaluation should prioritize metrics robust to imbalance (AUC-ROC, F1) over raw accuracy.